## Importing Libaries and Data

In [2]:
from sympy import *
import numpy as np
import matplotlib.pyplot as plt
from helpers import *
from implementations import *

In [34]:
x_train, x_test, y_train, train_ids, test_ids = load_csv_data("dataset")
y_train[y_train == -1] = 0 # format for logistic regression

In [35]:
feature_names = np.genfromtxt("dataset/x_train.csv", delimiter=",", dtype=str, max_rows=1)
feature_names = feature_names[1:]

## Splitting the Data & Handeling Missing Values

#### Splitting the training data into a training and a test set

In [36]:
def split_data(x, y, ratio, seed=1):
    """
    Args:
        x: numpy array of shape (N,), N is the number of samples.
        y: numpy array of shape (N,).
        ratio: scalar in [0,1]
        seed: integer.

    Returns:
        x_tr: numpy array containing the train data.
        x_te: numpy array containing the test data.
        y_tr: numpy array containing the train labels.
        y_te: numpy array containing the test labels.
    """
    # set seed
    np.random.seed(seed)
    N = x.shape[0]

    shuffled_indices = np.random.permutation(N)
    cutoff = int(np.floor(ratio * N))

    train_indices = shuffled_indices[:cutoff]
    test_indices = shuffled_indices[cutoff:]

    x_tr, x_te = x[train_indices], x[test_indices]
    y_tr, y_te = y[train_indices], y[test_indices]

    return x_tr, x_te, y_tr, y_te

x_tr, x_te, y_tr, y_te = split_data(x_train, y_train, 0.8, seed=1)

#### Exploring missing values for each feature

In [ ]:
fraction_missing = np.isnan(x_tr).mean(axis=0)
keep = fraction_missing < 0.6 # we drop any columns with more than 60% missing values

x_tr = x_tr[:, keep]
x_te = x_te[:, keep]
kept_names = feature_names[keep]

median = np.nanmedian(x_tr, axis=0)
x_tr = np.where(np.isnan(x_tr), median, x_tr)
x_te = np.where(np.isnan(x_te), median, x_te)

#### Standardize from training data

In [38]:
mean = x_tr.mean(axis=0)
std = x_tr.std(axis=0)
std[std == 0] = 1

x_tr = (x_tr - mean) / std
x_te = (x_te - mean) / std

#### Adding a bias column (intercept term, w0)

In [39]:
x_tr = np.c_[np.ones(x_tr.shape[0]), x_tr]
x_te = np.c_[np.ones(x_te.shape[0]), x_te]

feature_name_list = np.array(["bias"] + list(kept_names))

## Exploratory Analysis

In [40]:
N = y_tr.shape[0] + y_te.shape[0] 
total_disease = (y_tr == 1).sum() + (y_te == 1).sum()

print(f"Percentage of Patients with MICHD in the training set: {total_disease / N * 100:.2f} %")

Percentage of Patients with MICHD in the training set: 8.83 %


## Implementing Regularized Logistic Regression

In [41]:
y_tr = y_tr.reshape(-1, 1)
y_te = y_te.reshape(-1, 1)

#### Finding the best learning rate (gamma)

In [42]:
initial_w = np.zeros((x_tr.shape[1], 1))
max_iters = 30
lambda_ = 0
gammas = [1, 1.25, 1.5, 1.75, 2]

min_loss = np.inf
best_gamma = None

for gamma in gammas:
    w, loss = reg_logistic_regression(y_tr, x_tr, lambda_, initial_w, max_iters, gamma)
    if min_loss > loss:
        min_loss = loss
        best_gamma = gamma
    print(f"gamma = {gamma:<8} loss = {loss:.4f}")

print(f"Smallest Loss ={min_loss: .2f} Best gamma = {best_gamma}")

gamma = 1        loss = 0.2341
gamma = 1.25     loss = 0.2313
gamma = 1.5      loss = 0.2310
gamma = 1.75     loss = 0.2321
gamma = 2        loss = 0.2353
Smallest Loss = 0.23 Best gamma = 1.5


#### Finding the best hyperparameter (lambda)

In [43]:
lambdas = [0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2]

losses_tr, losses_te = [], []
w_list = []

for lam in lambdas:
    w, loss_tr = reg_logistic_regression(y_tr, x_tr, lam, initial_w, max_iters, best_gamma)
    loss_te = (1/len(y_te)) * np.sum(-y_te * (x_te @ w) + np.log(1 + np.exp(x_te @ w)))

    w_list.append(w)
    losses_tr.append(loss_tr)
    losses_te.append(loss_te)

    print(f"lambda = {lam:<8} train = {loss_tr:.4f} test = {loss_te:.4f}")

index = int(np.argmin(losses_te))
best_lambda = lambdas[index]
best_w = w_list[index]
print("best lambda:", best_lambda)

lambda = 0        train = 0.2310 test = 0.2283
lambda = 1e-06    train = 0.2310 test = 0.2283
lambda = 1e-05    train = 0.2310 test = 0.2283
lambda = 0.0001   train = 0.2310 test = 0.2283
lambda = 0.001    train = 0.2311 test = 0.2284
lambda = 0.01     train = 0.2326 test = 0.2297
best lambda: 0


#### Adding Interaction Features

In [ ]:
# Use only the m most influential features
m = 20
best_w = best_w.ravel()
feature_name_list = feature_name_list.ravel()

strengths = []
for col in range(1, len(best_w)):
    strengths.append((abs(best_w[col]), col)) # the strength of a feature’s weight, that feature’s column number

strengths.sort(reverse=True)

top_features = []
for strength, col in strengths[:m]:
    top_features.append(col)
    print(f"{feature_name_list[col]:<12} weight = {best_w[col]:+.3f}")

GENHLTH      weight = +0.340
SEX          weight = -0.253
TOLDHI2      weight = -0.204
_RFCHOL      weight = +0.195
BPHIGH4      weight = -0.186
_AGE80       weight = +0.178
_AGEG5YR     weight = +0.175
CVDSTRK3     weight = -0.133
EMPLOY1      weight = +0.122
_RFHYPE5     weight = +0.120
DIABETE3     weight = -0.101
SMOKE100     weight = -0.085
_AGE_G       weight = +0.078
CHCCOPD1     weight = -0.076
PNEUVAC3     weight = -0.075
CHOLCHK      weight = -0.070
PHYSHLTH     weight = -0.070
_RFSMOK3     weight = +0.068
VETERAN3     weight = -0.065
_DRDXAR1     weight = -0.063


[24,
 46,
 36,
 108,
 32,
 122,
 120,
 37,
 53,
 106,
 45,
 66,
 123,
 41,
 91,
 35,
 25,
 134,
 52,
 112]